In [11]:
import pandas as pd
from sqlalchemy import create_engine, text

In [12]:
connection_string = (
"mssql+pyodbc://./SalesPredictionDB"
    "?trusted_connection=yes"
"&driver=odbc+Driver+17+for+SQL+Server"
)

In [13]:
engine = create_engine(connection_string)

In [14]:
with engine.connect() as connection:
    result = connection.execute(text("SELECT DB_NAME()"))
    print(result.scalar())

SalesPredictionDB


In [15]:
df_clean = pd.read_csv("G:\ecommerce-sales-prediction\data\processed/online_retail_clean.csv")
df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [24]:
df_clean["Quantity"].dtype

dtype('int64')

In [18]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

In [22]:
customers = df_clean[["CustomerID", "Country"]].drop_duplicates(subset=["CustomerID"]).reset_index(drop=True)
products = df_clean[["StockCode", "Description"]].drop_duplicates(subset=["StockCode"]).reset_index(drop=True)
invoices = df_clean[["InvoiceNo", "CustomerID", "InvoiceDate"]].drop_duplicates(subset=["InvoiceNo"]).reset_index(drop=True)
invoice_items = df_clean[["InvoiceNo", "StockCode", "Quantity", "UnitPrice", "Revenue"]].copy()

In [23]:
customers.to_sql("Customers", con=engine, if_exists="append", index=False)
products.to_sql("Products", con=engine, if_exists="append", index=False)
invoices.to_sql("Invoices", con=engine, if_exists="append", index=False)
invoice_items.to_sql("InvoiceItems", con=engine, if_exists="append", index=False)

-1

In [28]:
with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT TOP 10
            i.InvoiceNo,
            i.CustomerID,
            c.Country,
            i.InvoiceDate
        FROM Invoices i
        JOIN Customers c
            ON i.CustomerID = c.CustomerID
    """))

    for row in result:
        print(row)

('536365', 17850, 'United Kingdom', datetime.datetime(2010, 12, 1, 8, 26))
('536366', 17850, 'United Kingdom', datetime.datetime(2010, 12, 1, 8, 28))
('536367', 13047, 'United Kingdom', datetime.datetime(2010, 12, 1, 8, 34))
('536368', 13047, 'United Kingdom', datetime.datetime(2010, 12, 1, 8, 34))
('536369', 13047, 'United Kingdom', datetime.datetime(2010, 12, 1, 8, 35))
('536370', 12583, 'France', datetime.datetime(2010, 12, 1, 8, 45))
('536371', 13748, 'United Kingdom', datetime.datetime(2010, 12, 1, 9, 0))
('536372', 17850, 'United Kingdom', datetime.datetime(2010, 12, 1, 9, 1))
('536373', 17850, 'United Kingdom', datetime.datetime(2010, 12, 1, 9, 2))
('536374', 15100, 'United Kingdom', datetime.datetime(2010, 12, 1, 9, 9))
